### The transformation Logic

In [0]:
%sql
SELECT 
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM workspace.silver.crm_sales sd
LEFT JOIN workspace.gold.dim_products pr
ON trim(upper(sd.product_number)) = trim(upper(pr.product_number))
LEFT JOIN workspace.gold.dim_customers cu
ON sd.customer_id = cu.customer_id ;


In [0]:
query ="""
SELECT 
 sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM workspace.silver.crm_sales sd
LEFT JOIN workspace.gold.dim_products pr
ON trim(upper(sd.product_number)) = trim(upper(pr.product_number))
LEFT JOIN workspace.gold.dim_customers cu
ON sd.customer_id = cu.customer_id ;
"""
df = spark.sql(query)

## Sanity Checks 

In [0]:
print("===== FACT TABLE QUALITY CEHCKS======")
total =df.count()
import pyspark.sql.functions as F
#How many sales have no matching produc??
no_product =df.filter(F.col("product_key").isNull()).count()
print(f"\n Sales with no product match:{no_product} /{total}({100*no_product//total}%)")
# Check 2: How many sales have no matching customer?
no_customer=df.filter(F.col("customer_key").isNull()).count()
print(f"\n sales with no customer match :{no_customer} /{total}({100*no_customer//total}%)")

df.select(
    F.min("order_date").alias("earliest_order"),
    F.max("order_date").alias("latest_order")
).display()
#Check 4: Total revenue
df.select(
    F.round(F.sum("sales_amount"),2).alias("total_revenue"),
    F.round(F.avg("sales_amount"),2).alias("avg_order_value"),
    F.count("order_number").alias("Total_orders")
).display()
# check 5: No negative sales amount
negative_sales=df.filter(F.col("sales_amount") < 0).count()
print(F"Rows with negative sales ammount: {negative_sales}(should be 0)")

### WRITING INTO GOLD TABLE

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.fact_sales")
print(f"Written {df.count()} rows to workspace.gold.fact_sales")


SANITY CHECKS

In [0]:
%sql
SELECT * FROM workspace.gold.fact_sales 